# Scaled dot-product attention
Why the scores are divided by sqrt(d_k): the spread of dot products grows with the dimension, a wide spread saturates the softmax, and dividing by sqrt(d_k) brings the variance back to 1. Then the full formula by hand against Keras' `MultiHeadAttention`.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
os.environ["CUDA_VISIBLE_DEVICES"] = ""       # no training here: the CPU is enough
import numpy as np
import pandas as pd
import keras
np.set_printoptions(precision=4, suppress=True)

def softmax(s, axis=-1):
    e = np.exp(s - s.max(axis=axis, keepdims=True))   # subtracting the max changes nothing, avoids overflow
    return e / e.sum(axis=axis, keepdims=True)

## One row of scores, without and with scaling (d_k = 3)
The query of "bank" against the keys of "money", "bank", "grows".

In [2]:
q_bank = np.array([1.0, 2.0, 1.0])
K = np.array([[2.0, 1.0, 0.0],     # key of money
              [1.0, 2.0, 1.0],     # key of bank
              [0.0, 1.0, 0.0]])    # key of grows
s = K @ q_bank
print("scores           ", s)
print("softmax          ", softmax(s))
print("scores / sqrt(3) ", s / np.sqrt(3))
print("softmax scaled   ", softmax(s / np.sqrt(3)))

scores            [4. 6. 2.]
softmax           [0.1173 0.8668 0.0159]
scores / sqrt(3)  [2.3094 3.4641 1.1547]
softmax scaled    [0.2228 0.707  0.0702]


## Dot products spread out as the dimension grows
1000 pairs of vectors with independent standard normal components (mean 0, variance 1), for d = 3, 100, 1000.

In [3]:
rng = np.random.default_rng(0)
rows = []
for d in (3, 100, 1000):
    q, k = rng.standard_normal((2, 1000, d))
    dot = (q * k).sum(1)
    print(f"d = {d:4d}: dot products from {dot.min():7.1f} to {dot.max():6.1f}, variance {dot.var():7.1f}, "
          f"after dividing by sqrt(d): variance {(dot / np.sqrt(d)).var():.2f}")
    rows += [dict(d=d, dot=x, scaled=x / np.sqrt(d)) for x in dot]
pd.DataFrame(rows).to_csv("data/dot_products.csv", index=False, float_format="%.4f")

d =    3: dot products from    -7.2 to    5.8, variance     2.9, after dividing by sqrt(d): variance 0.95
d =  100: dot products from   -27.6 to   32.4, variance    96.7, after dividing by sqrt(d): variance 0.97


d = 1000: dot products from  -105.7 to   95.8, variance  1015.6, after dividing by sqrt(d): variance 1.02


Variance of q.k for many dimensions (20,000 pairs each): it grows like d; after dividing by sqrt(d) it stays at 1.

In [4]:
dims = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
var = []
for d in dims:
    q, k = rng.standard_normal((2, 20000, d))
    dot = (q * k).sum(1)
    var.append(dict(d=d, var_dot=dot.var(), var_scaled=(dot / np.sqrt(d)).var()))
var = pd.DataFrame(var)
var.to_csv("data/variance_vs_d.csv", index=False, float_format="%.4f")
print(var.round(2).to_string(index=False))

   d  var_dot  var_scaled
   1     0.99        0.99
   2     1.96        0.98
   4     4.00        1.00
   8     7.99        1.00
  16    15.53        0.97
  32    32.29        1.01
  64    64.18        1.00
 128   128.09        1.00
 256   253.00        0.99
 512   509.44        1.00
1024  1048.09        1.02


## The rule Var(cX) = c^2 Var(X), on the numbers 10, 20, ..., 70

In [5]:
x = np.arange(10, 80, 10)
print("variance", x.var(), "  after dividing by 10:", (x / 10).var())

variance 400.0   after dividing by 10: 4.0


## A wide spread saturates the softmax
One query against 10 keys, 2000 random trials per dimension. For each softmax we record the largest weight and the size (Frobenius norm) of its Jacobian diag(a) - a a^T: the gradient that flows back through the softmax.

In [6]:
sat = []
for d in dims:
    q = rng.standard_normal((2000, 1, d))
    K = rng.standard_normal((2000, 10, d))
    s = (q * K).sum(-1)                       # 2000 x 10 scores
    for name, sc in (("unscaled", s), ("scaled", s / np.sqrt(d))):
        a = softmax(sc)
        J = a[:, :, None] * np.eye(10) - a[:, :, None] * a[:, None, :]
        sat.append(dict(d=d, version=name, max_weight=a.max(1).mean(),
                        jacobian_norm=np.linalg.norm(J, axis=(1, 2)).mean()))
sat = pd.DataFrame(sat)
sat.to_csv("data/saturation.csv", index=False, float_format="%.6f")
print(sat.pivot(index="d", columns="version", values=["max_weight", "jacobian_norm"]).round(4))

        max_weight          jacobian_norm         
version     scaled unscaled        scaled unscaled
d                                                 
1           0.2660   0.2660        0.3276   0.3276
2           0.2868   0.3628        0.3336   0.3379
4           0.2999   0.4855        0.3409   0.3401
8           0.3075   0.6038        0.3429   0.3169
16          0.3117   0.7207        0.3457   0.2689
32          0.3192   0.8060        0.3473   0.2104
64          0.3200   0.8633        0.3467   0.1642
128         0.3206   0.9049        0.3468   0.1194
256         0.3229   0.9289        0.3467   0.0909
512         0.3156   0.9494        0.3484   0.0676
1024        0.3212   0.9669        0.3485   0.0459


## Attention weights on a real sentence, d_k = 512, untrained (random) weights
The first 10 words of the first IMDB training review. Embeddings and W_Q, W_K are random, as at the start of training: embeddings N(0, 1), weights N(0, 1/d), so every component of q and k has variance about 1.

In [7]:
word_index = keras.datasets.imdb.get_word_index()
inv = {i + 3: w for w, i in word_index.items()}
(x_train, _), _ = keras.datasets.imdb.load_data()
words = [inv[i] for i in x_train[0][1:11]]
print(words)
d = 512
X = rng.standard_normal((10, d))
Wq, Wk = rng.normal(0, 1 / np.sqrt(d), (2, d, d))
Q, Kmat = X @ Wq, X @ Wk
S = Q @ Kmat.T
A_raw, A_scaled = softmax(S), softmax(S / np.sqrt(d))
print("variance of the scores:", S.var().round(1), "  after scaling:", (S / np.sqrt(d)).var().round(2))
print("largest weight per row, unscaled:", A_raw.max(1).round(2))
print("largest weight per row, scaled:  ", A_scaled.max(1).round(2))
print("rows with a weight above 0.99: unscaled", (A_raw.max(1) > 0.99).sum(), " scaled", (A_scaled.max(1) > 0.99).sum())
pd.DataFrame(A_raw, index=words, columns=words).to_csv("data/weights_unscaled.csv", float_format="%.4f")
pd.DataFrame(A_scaled, index=words, columns=words).to_csv("data/weights_scaled.csv", float_format="%.4f")

['this', 'film', 'was', 'just', 'brilliant', 'casting', 'location', 'scenery', 'story', 'direction']
variance of the scores: 658.9   after scaling: 1.29
largest weight per row, unscaled: [0.66 1.   1.   1.   0.94 1.   0.97 1.   0.56 1.  ]
largest weight per row, scaled:   [0.22 0.33 0.47 0.26 0.25 0.3  0.2  0.42 0.3  0.63]
rows with a weight above 0.99: unscaled 6  scaled 0


## The full formula by hand equals Keras' `MultiHeadAttention` with one head
"money bank grows", embeddings of size 8. Keras adds an output matrix after the attention (used by multi-head attention); we set it to the identity so that it changes nothing.

In [8]:
keras.utils.set_random_seed(0)
X = rng.standard_normal((1, 3, 8)).astype("float32")      # a batch of 1 sentence, 3 words, 8 numbers each
mha = keras.layers.MultiHeadAttention(num_heads=1, key_dim=8, use_bias=False)
mha(X, X)                                                   # builds the layer
Wq, Wk, Wv, Wo = mha.get_weights()
print("weight shapes:", Wq.shape, Wk.shape, Wv.shape, Wo.shape)
mha.set_weights([Wq, Wk, Wv, np.eye(8, dtype="float32")[None]])
out_keras, w_keras = mha(X, X, return_attention_scores=True)

x = X[0]
Q, K, V = x @ Wq[:, 0, :], x @ Wk[:, 0, :], x @ Wv[:, 0, :]
A = softmax(Q @ K.T / np.sqrt(8))
Y = A @ V
print("weights by hand:\n", A)
print("weights from Keras:\n", w_keras.numpy()[0, 0])
print("output by hand:\n", Y)
print("output from Keras:\n", out_keras.numpy()[0])
assert np.allclose(A, w_keras.numpy()[0, 0], atol=1e-5)
assert np.allclose(Y, out_keras.numpy()[0], atol=1e-5)
print("match: largest difference", np.abs(Y - out_keras.numpy()[0]).max())

weight shapes: (8, 1, 8) (8, 1, 8) (8, 1, 8) (1, 8, 8)
weights by hand:
 [[0.0146 0.6695 0.3159]
 [0.1586 0.5745 0.2669]
 [0.2502 0.1164 0.6335]]
weights from Keras:
 [[0.0146 0.6695 0.3159]
 [0.1586 0.5745 0.2669]
 [0.2502 0.1164 0.6335]]
output by hand:
 [[-1.3128  0.179  -0.1598 -0.4702 -0.3313 -0.2742  0.0284  0.2129]
 [-1.2909  0.4596 -0.1137 -0.6216 -0.1381 -0.2252  0.1135  0.0902]
 [-0.5874  0.5052  0.4854 -0.2668  0.3563 -0.9766  0.4793 -0.0501]]
output from Keras:
 [[-1.3128  0.179  -0.1598 -0.4702 -0.3313 -0.2742  0.0284  0.2129]
 [-1.2909  0.4596 -0.1137 -0.6216 -0.1381 -0.2252  0.1135  0.0902]
 [-0.5874  0.5052  0.4854 -0.2668  0.3563 -0.9766  0.4793 -0.0501]]
match: largest difference 2.3148678218731789e-07
